# 대규모 언어 모델 직접 배우기: LLM 기반 스테가노그래피(Steganography)
소개: 대규모 언어 모델을 활용한 텍스트 스테가노그래피 기법
> 텍스트 스테가노그래피는 자연스러운 텍스트 안에 비밀 정보를 숨기는 기술입니다. 이 튜토리얼에서는 GPT-2 모델을 활용하여 허프만 코딩과 고정 길이 코딩(FLC) 기반의 스테가노그래피를 구현해봅니다!

## 1. 이 튜토리얼의 목표:

- LLM 기반 텍스트 스테가노그래피의 원리 이해
- 허프만 코딩(Huffman Coding)을 활용한 스테가노그래피 구현
- 고정 길이 코딩(Fixed-Length Coding, FLC)을 활용한 스테가노그래피 구현
- 스테가노그래피 텍스트와 일반 텍스트의 비교

## 2. 배경 지식

### 2.1 텍스트 스테가노그래피란?

텍스트 스테가노그래피(Text Steganography)는 자연스러운 텍스트 안에 비밀 메시지(비트열)를 숨기는 기술입니다. 암호화와 달리, 스테가노그래피는 비밀 통신이 이루어지고 있다는 사실 자체를 숨기는 것이 목적입니다.

### 2.2 LLM 기반 스테가노그래피의 원리

LLM 기반 스테가노그래피는 언어 모델의 토큰 예측 확률 분포를 활용합니다:

1. **인코딩(숨기기)**: 언어 모델이 다음 토큰을 예측할 때, 확률이 높은 상위 k개의 토큰 후보를 선택합니다. 이 후보들에 이진 코드를 할당하고, 숨기려는 비트열에 맞는 토큰을 선택하여 텍스트를 생성합니다.
2. **디코딩(추출)**: 생성된 텍스트의 각 토큰에 대해 동일한 과정을 역으로 수행하여 숨겨진 비트열을 복원합니다.

### 2.3 코딩 방식

- **허프만 코딩(Huffman Coding)**: 토큰의 확률에 따라 가변 길이 코드를 할당합니다. 확률이 높은 토큰에는 짧은 코드를, 확률이 낮은 토큰에는 긴 코드를 할당하여 효율적인 정보 은닉이 가능합니다.
- **고정 길이 코딩(FLC)**: 상위 2^k개의 토큰에 동일한 길이(k비트)의 코드를 할당합니다. 구현이 간단하지만 허프만 코딩보다 효율이 낮을 수 있습니다.

## 3. 환경 설정

필요한 라이브러리를 설치합니다:
```
pip install torch transformers
```

## 4. 구현

### 4.1 라이브러리 임포트 및 유틸리티 함수 정의

In [ ]:
import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import random
from copy import deepcopy

# 상수 정의
EOS = 50256  # GPT-2의 EOS(문장 끝) 토큰 ID
UNK = 50256  # GPT-2에는 별도의 UNK 토큰이 없으므로 EOS를 사용

# 유틸리티 함수: 텍스트를 파일로 저장
def save_text(texts, path):
    with open(path, 'w', encoding='utf-8') as f:
        for text in texts:
            f.write(text + '\n')

### 4.2 허프만 트리 및 코딩 구현

허프만 코딩은 데이터 압축에 널리 사용되는 알고리즘입니다. 여기서는 토큰의 예측 확률을 기반으로 허프만 트리를 구성하고, 각 토큰에 이진 코드를 할당합니다.

- `Node`: 허프만 트리의 노드를 나타내는 클래스
- `createHuffmanTree`: 빈도(확률)를 기반으로 허프만 트리를 구성
- `huffmanEncoding`: 각 노드에 이진 코드를 할당

In [ ]:
# 허프만 트리 노드 클래스
class Node:
    def __init__(self, freq):
        self.left = None      # 왼쪽 자식 노드
        self.right = None     # 오른쪽 자식 노드
        self.father = None    # 부모 노드
        self.freq = freq      # 빈도(확률)
    def isLeft(self):
        return self.father.left == self

# 빈도 리스트로부터 노드 리스트 생성
def createNodes(freqs):
    return [Node(freq) for freq in freqs]

# 허프만 트리 구성
def createHuffmanTree(nodes):
    queue = nodes[:]
    while len(queue) > 1:
        queue.sort(key=lambda item:item.freq)  # 빈도 기준 정렬
        node_left = queue.pop(0)   # 가장 작은 빈도의 노드
        node_right = queue.pop(0)  # 두 번째로 작은 빈도의 노드
        node_father = Node(node_left.freq + node_right.freq)  # 부모 노드 생성
        node_father.left = node_left
        node_father.right = node_right
        node_left.father = node_father
        node_right.father = node_father
        queue.append(node_father)
    queue[0].father = None
    return queue[0]  # 루트 노드 반환

# 허프만 인코딩: 각 노드에 이진 코드 할당
def huffmanEncoding(nodes, root):
    codes = [''] * len(nodes)
    for i in range(len(nodes)):
        node_tmp = nodes[i]
        while node_tmp != root:
            if node_tmp.isLeft():
                codes[i] = '0' + codes[i]  # 왼쪽이면 '0'
            else:
                codes[i] = '1' + codes[i]  # 오른쪽이면 '1'
            node_tmp = node_tmp.father
    return codes

### 4.3 스테가노그래피 핸들러 구현

두 가지 스테가노그래피 방식을 구현합니다:

1. **Huffman 클래스**: 허프만 코딩 기반 스테가노그래피
   - 토큰 확률에 따라 가변 길이 코드를 할당하여 효율적으로 비트를 은닉
   - `extract()`: 주어진 토큰에서 숨겨진 비트를 추출
   - `__call__()`: 숨기려는 비트열에 맞는 토큰을 선택

2. **FLC 클래스**: 고정 길이 코딩 기반 스테가노그래피
   - 상위 2^k개 토큰에 k비트 고정 길이 코드를 할당
   - 구현이 간단하지만 허프만 코딩보다 비트 은닉 효율이 낮을 수 있음

In [ ]:
# 허프만 코딩 기반 스테가노그래피 핸들러
class Huffman():
    def __init__(self, k=None, bits=None):
        self.k = k        # 상위 k개 토큰 후보 수
        self.bits = bits   # 숨기려는 비트열
        
    def extract(self, pred_prob, word_index):
        """주어진 토큰에서 숨겨진 비트를 추출"""
        bit = None
        prob, idx = torch.topk(pred_prob, k=self.k, dim=1)
        word_prob = [[i.item(),j.item()] for i,j in zip(idx[0], prob[0])]
        nodes = createNodes([item[1] for item in word_prob])
        root = createHuffmanTree(nodes)
        codes = huffmanEncoding(nodes, root)
        for i, w_p in enumerate(word_prob):
            if w_p[0] == word_index:
                bit = codes[i]
                break
        return bit
    
    def __call__(self, pred_prob):
        """숨기려는 비트열에 맞는 토큰을 선택"""
        prob, idx = torch.topk(pred_prob, k=self.k, dim=1)
        word_prob = [[i.item(),j.item()] for i,j in zip(idx[0], prob[0])]
        nodes = createNodes([item[1] for item in word_prob])
        root = createHuffmanTree(nodes)
        codes = huffmanEncoding(nodes, root)
        for i,code in enumerate(codes):
            bit = self.bits[:len(code)]
            if bit == code:
                bit_word_index = word_prob[i][0]
                self.bits = self.bits[len(code):]  # 사용한 비트 제거
                break
        if self.extract(pred_prob,bit_word_index)!=bit:
            print("False")  # 검증 실패 시 경고
        bit_word_index = torch.LongTensor([bit_word_index]).to(pred_prob.device)
        return bit_word_index

In [ ]:
# 고정 길이 코딩(FLC) 기반 스테가노그래피 핸들러
class FLC():
    def __init__(self, k=None, bits=None):
        self.k = k        # 비트 수 (상위 2^k개 토큰 사용)
        self.bits = bits   # 숨기려는 비트열
    
    def extract(self, pred_prob, word_index):
        """주어진 토큰에서 숨겨진 비트를 추출"""
        bit = None
        prob, idx = torch.topk(pred_prob, k=2**self.k, dim=1)
        word_prob = [[i.item(),j.item()] for i,j in zip(idx[0], prob[0])]
        codes = [str(bin(i))[2:].zfill(self.k) for i in range(2**self.k)]  # 고정 길이 이진 코드 생성
        for i, w_p in enumerate(word_prob):
            if w_p[0] == word_index:
                bit = codes[i]
                break
        return bit
    
    def __call__(self, pred_prob):
        """숨기려는 비트열에 맞는 토큰을 선택"""
        prob, idx = torch.topk(pred_prob, k=2**self.k, dim=1)
        word_prob = [[i.item(),j.item()] for i,j in zip(idx[0], prob[0])]
        codes = [str(bin(i))[2:].zfill(self.k) for i in range(2**self.k)]
        bit = self.bits[:self.k]
        for i,code in enumerate(codes):
            if bit==code:
                bit_word_index = word_prob[i][0]
                self.bits = self.bits[self.k:]  # 사용한 비트 제거
                break
        bit_word_index = torch.LongTensor([bit_word_index]).to(pred_prob.device)
        return bit_word_index

### 4.4 텍스트 생성 함수

스테가노그래피 텍스트와 일반 텍스트를 동시에 생성하는 함수입니다.

- **스테가노그래피 텍스트**: 비밀 비트열을 숨기면서 자연스러운 텍스트를 생성
- **일반 텍스트**: 동일한 프롬프트에서 무작위 샘플링으로 생성한 비교용 텍스트

In [ ]:
# 랜덤 시드 설정 (재현성 보장)
def setup_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

# 스테가노그래피 텍스트 생성 함수
def generate_text_with_steganography(model, tokenizer, prompt, handle, max_length=30):
    device = model.device
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    
    # 스테가노그래피 텍스트와 일반 텍스트 초기화
    stega_output = input_ids.clone()
    normal_output = input_ids.clone()
    
    for _ in range(max_length):
        if len(handle.bits) == 0:
            break  # 숨길 비트가 더 이상 없으면 종료
            
        # 모델의 다음 토큰 예측 확률 계산
        outputs = model(stega_output)
        next_token_logits = outputs.logits[:, -1, :]
        next_token_probs = F.softmax(next_token_logits, dim=-1)
        
        # 스테가노그래피 적용: 비트열에 맞는 토큰 선택
        bit_word_index = handle(next_token_probs)
        stega_output = torch.cat([stega_output, bit_word_index.unsqueeze(1)], dim=1)
        
        # 일반 텍스트 생성: 무작위 샘플링
        outputs = model(normal_output)
        next_token_logits = outputs.logits[:, -1, :]
        next_token_probs = F.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(next_token_probs, num_samples=1)
        normal_output = torch.cat([normal_output, next_token], dim=1)
        
        if next_token[0].item() == tokenizer.eos_token_id:
            break  # EOS 토큰이 나오면 종료
    
    return stega_output, normal_output

## 5. 실행

GPT-2 모델을 로드하고, 랜덤 비트열을 생성한 후 스테가노그래피 텍스트를 생성합니다.

- `k=2`: 비트 파라미터 (허프만 코딩에서는 상위 2^k=4개 토큰 사용)
- `Huffman` 핸들러를 사용하여 스테가노그래피를 수행합니다
- `FLC` 핸들러로 변경하여 고정 길이 코딩 방식도 실험할 수 있습니다

In [ ]:
def main():
    setup_seed(2023)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    
    # GPT-2 모델과 토크나이저 로드
    model_name = "gpt2"
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
    
    # 랜덤 비트열 생성 (숨기려는 비밀 메시지)
    bits = ''.join([str(random.choice(list(range(2)))) for i in range(3000)])
    
    # 스테가노그래피 핸들러 초기화
    k = 2
    handle = Huffman(k=2**k, bits=bits)  # 허프만 코딩 방식
    # handle = FLC(k=k, bits=bits)       # 고정 길이 코딩 방식 (주석 해제하여 사용)
    
    # 스테가노그래피 텍스트 생성
    prompt = "Once upon a time"
    stega_output, normal_output = generate_text_with_steganography(
        model, tokenizer, prompt, handle
    )
    
    # 결과 디코딩 및 저장
    stega_text = tokenizer.decode(stega_output[0], skip_special_tokens=True)
    normal_text = tokenizer.decode(normal_output[0], skip_special_tokens=True)
    
    save_text([stega_text], './outputs-gpt2-stega.txt')
    save_text([normal_text], './outputs-gpt2-normal.txt')
    
    # 결과 출력
    print("=" * 60)
    print("스테가노그래피 텍스트 (비밀 비트가 숨겨진 텍스트):")
    print(stega_text)
    print("=" * 60)
    print("일반 텍스트 (비교용):")
    print(normal_text)
    print("=" * 60)

if __name__ == '__main__':
    main()

## 6. 정리

이 튜토리얼에서는 GPT-2 언어 모델을 활용한 텍스트 스테가노그래피를 구현했습니다.

### 핵심 요약:

- **텍스트 스테가노그래피**는 자연스러운 텍스트 안에 비밀 정보를 숨기는 기술입니다.
- **허프만 코딩** 방식은 토큰 확률에 따라 가변 길이 코드를 할당하여 높은 은닉 효율을 달성합니다.
- **고정 길이 코딩(FLC)** 방식은 구현이 간단하지만 은닉 효율이 상대적으로 낮습니다.
- 두 방식 모두 생성된 텍스트가 자연스러워 보이면서도 비밀 비트열을 정확하게 복원할 수 있습니다.

### 더 알아보기:

- 다양한 `k` 값을 실험하여 은닉 용량과 텍스트 품질 간의 트레이드오프를 관찰해보세요.
- `Huffman`과 `FLC` 핸들러를 교체하여 두 방식의 차이를 비교해보세요.
- 더 큰 언어 모델(GPT-2 Medium, Large 등)을 사용하면 텍스트 품질이 어떻게 변하는지 확인해보세요.